# FAISS index and ANN recall

Issue: #26

## Question

Exact nearest-neighbour search compares a query with every item. An approximate index (ANN) is faster and smaller but can miss some true neighbours. How much do we lose, and what do we gain? Then: what does the Two-Tower add to the candidate pool?

## Data used

Inputs: the Two-Tower from #24 (`data/models/two_tower/`: item embeddings, 394,663 x 64, unit length, and the query tower), the ranking queries (#10), and the co-visitation and popularity retrievers from #21/#22.

Outputs: `data/models/faiss.index` (the chosen index) and the tables below. Everything runs in process on the Mac, no server. (PyTorch and FAISS cannot be imported in one process on macOS, because each brings its own OpenMP runtime. The query vectors are exported by a separate script, `notebooks/tt_export_queries.py`, and this notebook only reads them.)

Three indexes over the item embeddings (inner product = cosine here):
- **Flat** (exact): compares with every item. The ground truth.
- **IVF-PQ**: items are grouped into 1,024 clusters (IVF) and stored as 16-byte compressed codes (PQ). A query only visits `nprobe` clusters.
- **HNSW**: a graph of neighbours; a query walks the graph. `efSearch` sets how wide the walk is.

Recall is measured on 5,000 random evaluation queries as the share of the exact top 100 that the ANN index also returns.

## Rule for choosing the index (written before looking at results)

Among the tested settings with **recall@100 of at least 0.95**, choose the one with the lowest **p95 latency**; if two are within 10 percent on latency, prefer the smaller index. If none reaches 0.95, choose the highest recall.

In [ ]:
import json
import os
import resource
import subprocess
import sys
import time
from pathlib import Path

import faiss
import numpy as np
import polars as pl
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from evalset import prepare_eval
from features import AsOf
from metrics import item_gains, ndcg_at_k, recall_at_k
from pool import build_pool
from split import day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
D14 = day_to_ms(14)
TT = DATA / "models" / "two_tower"
FAISS_DIR = DATA / "models" / "faiss"
FAISS_DIR.mkdir(parents=True, exist_ok=True)

cfg = json.loads((TT / "config.json").read_text())
catalog_aids = np.load(TT / "catalog_aids.npy")
vecs = np.ascontiguousarray(np.load(TT / "item_embeddings.npy"), dtype=np.float32)
n_items, dim = vecs.shape
print("item embeddings:", vecs.shape, "norms", float(np.linalg.norm(vecs, axis=1).min()), "to", float(np.linalg.norm(vecs, axis=1).max()))
assert n_items == cfg["n_items"] and np.allclose(np.linalg.norm(vecs, axis=1), 1, atol=1e-4)

# PyTorch and FAISS cannot share a process on macOS (two OpenMP runtimes). The query vectors are
# written by a separate process (notebooks/tt_export_queries.py) and read here as plain arrays.
ROOT = DATA.parent
subprocess.run([sys.executable, str(ROOT / "notebooks" / "tt_export_queries.py")], check=True)
qvec = np.load(TT / "eval_query_vectors.npy").astype(np.float32)
prepared, prep_stats = prepare_eval(pl.read_parquet(DATA / "eval" / "ranking_queries.parquet"))
assert np.load(TT / "eval_query_sessions.npy").tolist() == [q["session"] for q in prepared], "query order differs"
e_cats = np.array([q["category"] for q in prepared])
assert np.allclose(np.linalg.norm(qvec, axis=1), 1, atol=1e-4)
print("query vectors:", qvec.shape)

## Build the indexes

In [ ]:
def build_flat():
    ix = faiss.IndexFlatIP(dim)
    ix.add(vecs)
    return ix


def build_ivfpq(nlist=1024, m=16, nbits=8):
    quant = faiss.IndexFlatIP(dim)
    ix = faiss.IndexIVFPQ(quant, dim, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT)
    rng = np.random.default_rng(0)
    ix.train(vecs[rng.choice(n_items, size=100_000, replace=False)])
    ix.add(vecs)
    return ix


def build_hnsw(m=32, ef_construction=200):
    ix = faiss.IndexHNSWFlat(dim, m, faiss.METRIC_INNER_PRODUCT)
    ix.hnsw.efConstruction = ef_construction
    ix.add(vecs)
    return ix


built = {}
for name, fn in (("Flat (exact)", build_flat), ("IVF-PQ", build_ivfpq), ("HNSW", build_hnsw)):
    t = time.time()
    built[name] = (fn(), time.time() - t)
    print(f"{name}: built in {built[name][1]:.1f} s")
flat = built["Flat (exact)"][0]
for name, (ix, _) in built.items():
    assert ix.ntotal == n_items
    path = FAISS_DIR / (name.split()[0].lower().replace("-", "") + ".index")
    faiss.write_index(ix, str(path))

## Recall against exact search, latency and size

In [ ]:
rng = np.random.default_rng(0)
bench = rng.choice(len(prepared), size=5000, replace=False)
Q = np.ascontiguousarray(qvec[bench])
_, exact_ids = flat.search(Q, 100)


def recall_vs_exact(ids):
    return float(np.mean([len(set(a.tolist()) & set(b.tolist())) / 100 for a, b in zip(ids, exact_ids, strict=True)]))


def latency(ix, n=1000):
    faiss.omp_set_num_threads(1)
    for i in range(50):                       # warm up
        ix.search(Q[i : i + 1], 100)
    times = []
    for i in range(n):
        t = time.perf_counter()
        ix.search(Q[i : i + 1], 100)
        times.append((time.perf_counter() - t) * 1000)
    faiss.omp_set_num_threads(os.cpu_count())
    return float(np.percentile(times, 50)), float(np.percentile(times, 95))


configs = [("Flat (exact)", "-", lambda ix: None),
           ("IVF-PQ", "nprobe=8", lambda ix: setattr(ix, "nprobe", 8)),
           ("IVF-PQ", "nprobe=32", lambda ix: setattr(ix, "nprobe", 32)),
           ("IVF-PQ", "nprobe=128", lambda ix: setattr(ix, "nprobe", 128)),
           ("HNSW", "efSearch=64", lambda ix: setattr(ix.hnsw, "efSearch", 64)),
           ("HNSW", "efSearch=128", lambda ix: setattr(ix.hnsw, "efSearch", 128)),
           ("HNSW", "efSearch=256", lambda ix: setattr(ix.hnsw, "efSearch", 256))]
rows = []
for name, params, setter in configs:
    ix, build_s = built[name]
    setter(ix)
    _, ids = ix.search(Q, 100)
    p50, p95 = latency(ix)
    size = os.path.getsize(FAISS_DIR / (name.split()[0].lower().replace("-", "") + ".index")) / 1e6
    rows.append({"index": name, "params": params, "recall@100": recall_vs_exact(ids), "p50_ms": p50, "p95_ms": p95, "size_MB": size, "build_s": build_s})
results = pl.DataFrame(rows)
with pl.Config(tbl_rows=10, tbl_width_chars=140):
    print(results)
assert results.filter(pl.col("index") == "Flat (exact)")["recall@100"][0] == 1.0

## Choose one (rule above)

In [ ]:
ok = results.filter((pl.col("recall@100") >= 0.95) & (pl.col("index") != "Flat (exact)"))
if ok.height:
    best_p95 = ok["p95_ms"].min()
    close = ok.filter(pl.col("p95_ms") <= best_p95 * 1.10).sort("size_MB")
    chosen = close.row(0, named=True)
    how = "recall@100 >= 0.95, lowest p95 latency (ties within 10% go to the smaller index)"
else:
    chosen = results.filter(pl.col("index") != "Flat (exact)").sort("recall@100", descending=True).row(0, named=True)
    how = "no index reached 0.95, highest recall"
print("chosen:", chosen["index"], chosen["params"], "-", how)
print(chosen)
name, params = chosen["index"], chosen["params"]
chosen_index = built[name][0]
{c[0] + c[1]: c[2] for c in configs}[name + params](chosen_index)
faiss.write_index(chosen_index, str(DATA / "models" / "faiss.index"))
print("saved data/models/faiss.index,", round(os.path.getsize(DATA / "models" / "faiss.index") / 1e6, 1), "MB")

## Add Two-Tower candidates to the shared pool (#21)

The pool is limited to 200 candidates and merged by ranks. The Two-Tower list comes from the **chosen ANN index**: search the whole catalog for the top 500 neighbours, keep those in the query category, drop prefix items, take 200. The same list from **exact in-category search** is shown for comparison, so the cost of ANN plus filtering is visible.

In [ ]:
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
m14 = sparse.load_npz(DATA / "covis" / "covis.npz")
ids14 = np.load(DATA / "covis" / "item_ids.npy")
asof = AsOf(events, D14, categories, m14, ids14)
cat_of_item = np.array([asof.item_cluster.get(int(a), -1) for a in catalog_aids])

# Two-Tower candidates: ANN (global top 500, then category filter) and exact (in-category)
t = time.time()
K_SEARCH = 500
_, ann_ids = chosen_index.search(np.ascontiguousarray(qvec), K_SEARCH)
print(f"ANN search of {len(qvec):,} queries, k={K_SEARCH}: {time.time() - t:.0f} s")
by_cat = {c: np.flatnonzero(cat_of_item == c) for c in range(cfg["n_cats"])}
tt_ann, tt_exact, in_cat_counts = {}, {}, []
for c in range(cfg["n_cats"]):
    rows_c = np.flatnonzero(e_cats == c)
    if not len(rows_c):
        continue
    members = by_cat[c]
    scores = qvec[rows_c] @ vecs[members].T
    top = np.argsort(-scores, axis=1)[:, :260]
    for r, ex in zip(rows_c, top, strict=True):
        prefix = prepared[r]["prefix"]
        tt_exact[r] = [int(a) for a in catalog_aids[members[ex]] if int(a) not in prefix][:200]
for r, q in enumerate(prepared):
    cand = [int(a) for a in catalog_aids[ann_ids[r][ann_ids[r] >= 0]]]
    keep = [a for a in cand if asof.item_cluster.get(a, -1) == q["category"] and a not in q["prefix"]]
    in_cat_counts.append(len(keep))
    tt_ann[r] = keep[:200]
print(f"ANN candidates in the query category per query: mean {np.mean(in_cat_counts):.1f}, median {int(np.median(in_cat_counts))}, share of queries with 200 or more {np.mean(np.array(in_cat_counts) >= 200):.1%}")

In [ ]:
KS = [20, 50, 100, 200]
cov_lists = [asof.covis.rank(q, pure=True) for q in prepared]
pop_lists = []
for q in prepared:
    lst = [int(a) for a in asof.pop_by_cat[q["category"]] if int(a) not in q["prefix"]][:200]
    pop_lists.append(lst)


def pool_recall(name, extra=None):
    recs = {k: [] for k in KS}
    best = []
    empty = 0
    for r, q in enumerate(prepared):
        lists = {"covis": cov_lists[r], "pop": pop_lists[r]}
        if extra is not None:
            lists["two_tower"] = extra[r]
        pool = build_pool(q["prefix"], lists, cap=200)
        aids = [c["aid"] for c in pool]
        relevant = set().union(*q["labels"].values())
        for k in KS:
            recs[k].append(recall_at_k(aids, relevant, k=k))
        gains = item_gains(q["labels"])
        in_pool = sorted((a for a in aids if a in gains), key=lambda a: -gains[a])
        best.append(ndcg_at_k(in_pool + [a for a in aids if a not in gains], q["labels"], k=10))
        empty += not in_pool
    out = {"pool": name, **{f"recall@{k}": float(np.mean(v)) for k, v in recs.items()}, "best_possible_ndcg10": float(np.mean(best)), "queries_without_positive": empty / len(prepared)}
    return out


pools = [pool_recall("covis + popularity (#21)"),
         pool_recall("+ Two-Tower via chosen ANN", tt_ann),
         pool_recall("+ Two-Tower exact in-category", tt_exact)]
pool_table = pl.DataFrame(pools)
with pl.Config(tbl_width_chars=200, tbl_cols=8):
    print(pool_table)

single = pl.DataFrame([{"list": name, **{f"recall@{k}": float(np.mean([recall_at_k(lst[r], set().union(*q["labels"].values()), k=k) for r, q in enumerate(prepared)])) for k in KS}}
                       for name, lst in (("Two-Tower ANN + filter", tt_ann), ("Two-Tower exact in-category", tt_exact), ("co-visitation (pure)", cov_lists), ("popularity", pop_lists))])
print(single)

In [ ]:
base, with_ann, with_exact = pools
print(f"pool recall@200: {base['recall@200']:.4f} -> {with_ann['recall@200']:.4f} with Two-Tower via ANN, {with_exact['recall@200']:.4f} with exact Two-Tower")
print(f"best possible NDCG@10: {base['best_possible_ndcg10']:.4f} -> {with_ann['best_possible_ndcg10']:.4f}")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

**Index comparison** (5,000 queries, recall = share of the exact top 100 returned; latency = one query at a time, one thread, warm):

| Index | Setting | recall@100 | p50 | p95 | Size |
|---|---|---|---|---|---|
| Flat (exact) | - | 1.000 | 3.61 ms | 4.23 ms | 101 MB |
| IVF-PQ | nprobe 8 / 32 / 128 | 0.878 / 0.878 / 0.878 | 0.06 / 0.16 / 0.37 ms | 0.07 / 0.19 / 0.41 ms | 9.8 MB |
| HNSW | efSearch 64 | 0.933 | 0.05 ms | 0.10 ms | 208 MB |
| HNSW | efSearch 128 | **0.985** | 0.08 ms | 0.16 ms | 208 MB |
| HNSW | efSearch 256 | 0.995 | 0.24 ms | 0.40 ms | 208 MB |

- **Chosen: HNSW with efSearch 128** by the rule written before the run (recall@100 at least 0.95, lowest p95 among those). It is saved as `data/models/faiss.index`.
- **IVF-PQ tops out at 0.878** whatever `nprobe` is: raising nprobe visits more clusters but the item codes are compressed (16 bytes each), and that quantisation error caps recall. It is 21 times smaller than HNSW (9.8 MB vs 208 MB) and the fastest, so it would be the choice if index size mattered more than the last 12 points of recall. Larger or more accurate codes (more sub-quantisers, or re-ranking with exact vectors) were not tried.
- **HNSW is bigger than exact search here** (208 MB vs 101 MB): it stores the full vectors plus the graph. It is about 25 to 50 times faster than exact (0.16 ms vs 4.2 ms p95). At 395,000 items exact search is already fast enough, so ANN is a lesson in the trade-off, not a necessity at this scale. Restricting exact search to the items of the query category (about 5,000) is cheaper still.

**Two-Tower candidates in the pool** (56,544 queries, pool cap 200, merged by rank):

| Pool | recall@20 | recall@50 | recall@100 | recall@200 | best possible NDCG@10 | queries with no positive |
|---|---|---|---|---|---|---|
| covis + popularity (#21) | 0.339 | 0.458 | 0.539 | **0.608** | 0.621 | 33.3% |
| + Two-Tower via the chosen ANN index | 0.191 | 0.299 | 0.493 | 0.596 | 0.609 | 34.3% |
| + Two-Tower exact in-category | 0.189 | 0.298 | 0.493 | 0.597 | 0.610 | 34.2% |

- **The Two-Tower does not improve the pool; recall@200 goes down slightly** (0.608 to 0.596). The pool is capped at 200 and merged by rank fusion, and the Two-Tower list is much like the popularity list (its own recall@200 is 0.360 against 0.378 for popularity), so it takes slots from candidates that already found relevant items without bringing many new ones.
- **The recall at small K drops a lot** (recall@20 from 0.339 to 0.191) because fusion now mixes three lists in the top ranks. That does not matter for a reranker, which reorders the whole pool, but it means the fused order is not a ranking.
- **ANN costs almost nothing** compared with exact search: 0.596 vs 0.597 pool recall@200, and 0.360 vs 0.366 for the list on its own. Searching the whole catalog for 500 neighbours and keeping the query category gives 345 in-category candidates on average (98.9% of queries have 200 or more), because the query vector already contains the category.
- **Consequence for the plan:** the shared pool for #27 to #29 stays covis + popularity (the features in #22 were built on it), and the Two-Tower can be used as a feature or as a method of its own.

PyTorch and FAISS cannot be imported in one process on macOS (two OpenMP runtimes crash the kernel); the query vectors are exported by `notebooks/tt_export_queries.py`.

## What I learned

To be written by the owner of the project.